# Lab 1: API smoke-test моделей

Ноутбук воспроизводит таблицу метрик и расчёт стоимости по четырём логам от 18.09.2026. Сетевых вызовов и секретов здесь нет.

In [ ]:
runs = [
    {
        'model': 'openai/gpt-5.6-sol',
        'total_time_s': 5.427549,
        'ttft_content_s': 4.638078,
        'prompt_tokens': 214,
        'completion_tokens': 252,
        'total_tokens': 466,
        'tokens_per_second': 46.430,
        'input_usd_per_million': 4.00,
        'output_usd_per_million': 20.00,
        'format': 'raw JSON; 10 expected fields',
    },
    {
        'model': 'mistralai/mistral-medium-3-5',
        'total_time_s': 1.338671,
        'ttft_content_s': 0.894906,
        'prompt_tokens': 274,
        'completion_tokens': 76,
        'total_tokens': 350,
        'tokens_per_second': 56.773,
        'input_usd_per_million': 1.50,
        'output_usd_per_million': 7.50,
        'format': 'plain text; not JSON',
    },
    {
        'model': 'qwen/qwen3.8-max',
        'total_time_s': 55.148368,
        'ttft_content_s': 52.433001,
        'prompt_tokens': 283,
        'completion_tokens': 2136,
        'total_tokens': 2419,
        'tokens_per_second': 38.732,
        'input_usd_per_million': 2.00,
        'output_usd_per_million': 6.00,
        'format': 'raw JSON; 10 expected fields',
    },
    {
        'model': 'deepseek/deepseek-v4-pro',
        'total_time_s': 27.688746,
        'ttft_content_s': 25.108124,
        'prompt_tokens': 243,
        'completion_tokens': 1540,
        'total_tokens': 1783,
        'tokens_per_second': 55.618,
        'input_usd_per_million': 1.32,
        'output_usd_per_million': 3.96,
        'format': 'JSON in Markdown; incompatible keys',
    },
]

for run in runs:
    assert run['prompt_tokens'] + run['completion_tokens'] == run['total_tokens']

len(runs)

In [ ]:
for run in runs:
    run['estimated_cost_usd'] = (
        run['prompt_tokens'] * run['input_usd_per_million']
        + run['completion_tokens'] * run['output_usd_per_million']
    ) / 1_000_000

columns = [
    'model', 'total_time_s', 'ttft_content_s', 'prompt_tokens',
    'completion_tokens', 'total_tokens', 'estimated_cost_usd', 'format'
]
print('| ' + ' | '.join(columns) + ' |')
print('| ' + ' | '.join(['---'] * len(columns)) + ' |')
for run in runs:
    values = [
        run['model'],
        f"{run['total_time_s']:.6f}",
        f"{run['ttft_content_s']:.6f}",
        str(run['prompt_tokens']),
        str(run['completion_tokens']),
        str(run['total_tokens']),
        f"{run['estimated_cost_usd']:.6f}",
        run['format'],
    ]
    print('| ' + ' | '.join(values) + ' |')

In [ ]:
by_latency = sorted(runs, key=lambda row: row['total_time_s'])
by_tokens = sorted(runs, key=lambda row: row['total_tokens'])
by_cost = sorted(runs, key=lambda row: row['estimated_cost_usd'])

print('Fastest:', by_latency[0]['model'])
print('Fewest tokens:', by_tokens[0]['model'])
print('Lowest estimated cost:', by_cost[0]['model'])
print('GPT/Qwen token ratio:', round(runs[2]['total_tokens'] / runs[0]['total_tokens'], 2))
print('GPT/Qwen latency ratio:', round(runs[2]['total_time_s'] / runs[0]['total_time_s'], 2))

## Вывод

Для следующего эксперимента выбран `openai/gpt-5.6-sol`: он вернул машинно-читаемый JSON при умеренных задержке и расходе. `mistralai/mistral-medium-3-5` остаётся резервом и должен быть повторно проверен с реальным `response_format: json_schema`.

Этот smoke-тест не измеряет качество разбора: в запросе не было обращения, истории, фрагментов БЗ и JSON Schema. Для оценки качества нужны 10 golden-примеров, не менее трёх повторов и автоматическая валидация схемы.